In [ ]:
# =========================================================
# Imports et configuration
# =========================================================
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

from config import RANDOM_STATE, TEST_SIZE, TARGET_REG, FEATURES_REG
from data import load_data, clean_data

sns.set_theme(style="whitegrid")

# =========================================================
# 1. Chargement des données
# =========================================================
df = load_data()
df_clean = clean_data(df)
print(f"Taille initiale : {len(df_clean)} morceaux")

# =========================================================
# 2. Modèle Simple A : Prédiction via 'Loudness'
# =========================================================
# Filtrage A : On exclut les durées aberrantes et les quasi-silences
df_loud = df_clean[
    (df_clean['duration_min'] >= 1.0) & (df_clean['duration_min'] <= 10.0) &
    (df_clean['loudness'] >= -20.0)
]
print(f"Taille après Filtre A (Loudness) : {len(df_loud)} morceaux")

X_loud = df_loud[['loudness']]
y_loud = df_loud[TARGET_REG]

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_loud, y_loud, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

model_simple_loud = LinearRegression().fit(X_train_l, y_train_l)

# =========================================================
# 3. Modèle Simple B : Prédiction via 'Acousticness'
# =========================================================
# Filtrage B : On exclut les contenus très parlés (podcasts)
df_aco = df_clean[
    (df_clean['duration_min'] >= 1.0) & (df_clean['duration_min'] <= 10.0) &
    (df_clean['speechiness'] < 0.5)
]
print(f"Taille après Filtre B (Acousticness) : {len(df_aco)} morceaux")

X_aco = df_aco[['acousticness']]
y_aco = df_aco[TARGET_REG]

X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(
    X_aco, y_aco, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

model_simple_aco = LinearRegression().fit(X_train_a, y_train_a)

# =========================================================
# 4. Modèle Multiple : Prédiction via toutes les variables
# =========================================================
# Filtrage Global : On cumule tous les filtres pour un jeu pur
df_multi = df_clean[
    (df_clean['duration_min'] >= 1.0) & (df_clean['duration_min'] <= 10.0) &
    (df_clean['loudness'] >= -20.0) & 
    (df_clean['tempo'] > 0) &
    (df_clean['speechiness'] < 0.5)
]
print(f"Taille après Filtre Global (Multiple) : {len(df_multi)} morceaux")

X_multi = df_multi[FEATURES_REG]
y_multi = df_multi[TARGET_REG]

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_multi, y_multi, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

model_multi = LinearRegression().fit(X_train_m, y_train_m)
y_pred_m = model_multi.predict(X_test_m)

# =========================================================
# 5. Évaluation et Comparaison Commune
# =========================================================
def get_metrics(name, model, X_te, y_te):
    pred = model.predict(X_te)
    return {
        'Modèle': name,
        'R²': round(r2_score(y_te, pred), 3),
        'MAE': round(mean_absolute_error(y_te, pred), 3),
        'RMSE': round(np.sqrt(mean_squared_error(y_te, pred)), 3)
    }

results_reg = pd.DataFrame([
    get_metrics("Simple (Loudness)", model_simple_loud, X_test_l, y_test_l),
    get_metrics("Simple (Acousticness)", model_simple_aco, X_test_a, y_test_a),
    get_metrics("Multiple (Toutes variables)", model_multi, X_test_m, y_test_m)
])

print("\n--- Performances des modèles sur leurs jeux de Test respectifs ---")
print(results_reg)

# =========================================================
# 6. Visualisations du modèle final (Multiple)
# =========================================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Nuage de points
ax1.scatter(y_test_m, y_pred_m, alpha=0.3, color='royalblue')
ax1.plot([0, 1], [0, 1], color='red', linestyle='--', linewidth=2)
ax1.set_title('Modèle Multiple : Énergie Réelle vs Prédite')
ax1.set_xlabel('Énergie Réelle (y_test)')
ax1.set_ylabel('Énergie Prédite (y_pred)')
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 1)

# Coefficients
coefficients = pd.DataFrame(
    list(zip(FEATURES_REG, model_multi.coef_)), 
    columns=['Variable', 'Coefficient']
)
coefficients['Impact Absolu'] = coefficients['Coefficient'].abs()
coefficients = coefficients.sort_values(by='Impact Absolu', ascending=False)

sns.barplot(data=coefficients, x='Coefficient', y='Variable', palette='vlag', ax=ax2)
ax2.set_title("Coefficients de la Régression Multiple")

plt.tight_layout()
plt.show()

# =========================================================
# 7. Sauvegarde du modèle final
# =========================================================
os.makedirs('../models', exist_ok=True)
joblib.dump(model_multi, '../models/regression.joblib')
print("\nModèle multiple sauvegardé dans '../models/regression.joblib'")

Taille initiale : 28356 morceaux
Taille après Filtre A (Loudness) : 28273 morceaux
Taille après Filtre B (Acousticness) : 28193 morceaux
Taille après Filtre Global (Multiple) : 28135 morceaux

--- Performances des modèles sur leurs jeux de Test respectifs ---
                        Modèle     R²    MAE   RMSE
0            Simple (Loudness)  0.455  0.107  0.134
1        Simple (Acousticness)  0.308  0.126  0.155
2  Multiple (Toutes variables)  0.632  0.087  0.110

Modèle multiple sauvegardé dans '../models/regression.joblib'
